In [53]:
import pandas as pd
from functools import reduce

paths = {
    "tabular": "../data/tabular_embeddings.csv",
    "temporal": "../data/temporal_embeddings.csv",
    "graph": "../data/graph_embeddings.csv"

}



In [54]:
dfs = {k: pd.read_csv(v) for k, v in paths.items()}

for k, df in dfs.items():
    print(k, df.shape)


tabular (15420, 20)
temporal (15420, 19)
graph (15420, 35)


In [55]:
df = reduce(
    lambda left, right: pd.merge(left, right, on=["claim_id", "fraudfound_p"]),
    dfs.values()
)

print("Merged shape:", df.shape)


Merged shape: (15420, 70)


In [56]:
drop_cols = ["claim_id", "fraudfound_p"]
bad_cols = [c for c in df.columns if "leaf" in c or "raw_score" in c]

df = df.drop(columns=drop_cols + bad_cols, errors="ignore")


In [57]:
df["avg_prob"] = (
    df["tabular_fraud_probability"] +
    df["temporal_fraud_probability"] +
    df["graph_fraud_probability"]
) / 3

df["max_prob"] = df[
    ["tabular_fraud_probability",
     "temporal_fraud_probability",
     "graph_fraud_probability"]
].max(axis=1)

df["min_prob"] = df[
    ["tabular_fraud_probability",
     "temporal_fraud_probability",
     "graph_fraud_probability"]
].min(axis=1)

# 🔥 weighted signal (MOST IMPORTANT)
df["weighted_score"] = (
    0.7 * df["tabular_fraud_probability"] +
    0.2 * df["graph_fraud_probability"] +
    0.1 * df["temporal_fraud_probability"]
)

# 🔥 interactions
df["tab_x_graph"] = df["tabular_fraud_probability"] * df["graph_fraud_probability"]
df["tab_x_temp"] = df["tabular_fraud_probability"] * df["temporal_fraud_probability"]

# 🔥 differences
df["tab_graph_diff"] = abs(
    df["tabular_fraud_probability"] - df["graph_fraud_probability"]
)

df["tab_temp_diff"] = abs(
    df["tabular_fraud_probability"] - df["temporal_fraud_probability"]
)

# 🔥 anchor strongest model
df["final_hint"] = df["tabular_fraud_probability"]


In [58]:

y = dfs["tabular"]["fraudfound_p"].values
X = df.values

In [59]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


In [60]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [61]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import roc_auc_score
import copy

class FusionDNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze()

In [62]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32)
y_test  = torch.tensor(y_test, dtype=torch.float32)

In [63]:
class_counts = np.bincount(y_train.numpy().astype(int))
pos_weight = torch.tensor([class_counts[0] / class_counts[1]])

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [64]:


model = FusionDNN(input_dim=X_train.shape[1])
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

best_auc = 0
best_model = copy.deepcopy(model.state_dict())
patience = 5
patience_counter = 0

for epoch in range(50):
    model.train()

    logits = model(X_train)
    loss = criterion(logits, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(X_test)).numpy()
        auc = roc_auc_score(y_test, probs)

    print(f"Epoch {epoch} | Loss {loss.item():.4f} | AUC {auc:.4f}")

    if auc > best_auc:
        best_auc = auc
        best_model = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print("Early stopping triggered")
        break

# load best model
model.load_state_dict(best_model)
print("🔥 Best AUC:", best_auc)

Epoch 0 | Loss 1.3683 | AUC 0.5018
Epoch 1 | Loss 1.3237 | AUC 0.5830
Epoch 2 | Loss 1.2823 | AUC 0.6472
Epoch 3 | Loss 1.2441 | AUC 0.6927
Epoch 4 | Loss 1.2087 | AUC 0.7238
Epoch 5 | Loss 1.1758 | AUC 0.7457
Epoch 6 | Loss 1.1451 | AUC 0.7608
Epoch 7 | Loss 1.1165 | AUC 0.7722
Epoch 8 | Loss 1.0897 | AUC 0.7810
Epoch 9 | Loss 1.0643 | AUC 0.7881
Epoch 10 | Loss 1.0403 | AUC 0.7941
Epoch 11 | Loss 1.0175 | AUC 0.7991
Epoch 12 | Loss 0.9958 | AUC 0.8036
Epoch 13 | Loss 0.9750 | AUC 0.8073
Epoch 14 | Loss 0.9550 | AUC 0.8105
Epoch 15 | Loss 0.9357 | AUC 0.8133
Epoch 16 | Loss 0.9170 | AUC 0.8159
Epoch 17 | Loss 0.8990 | AUC 0.8182
Epoch 18 | Loss 0.8815 | AUC 0.8202
Epoch 19 | Loss 0.8647 | AUC 0.8220
Epoch 20 | Loss 0.8486 | AUC 0.8237
Epoch 21 | Loss 0.8330 | AUC 0.8251
Epoch 22 | Loss 0.8182 | AUC 0.8263
Epoch 23 | Loss 0.8040 | AUC 0.8274
Epoch 24 | Loss 0.7905 | AUC 0.8283
Epoch 25 | Loss 0.7777 | AUC 0.8291
Epoch 26 | Loss 0.7656 | AUC 0.8297
Epoch 27 | Loss 0.7542 | AUC 0.8300
Ep

(15420, 70)


In [65]:
model.eval()
with torch.no_grad():
    fraud_scores = torch.sigmoid(model(X_test)).numpy()

print("Sample fraud scores:", fraud_scores[:10])

Sample fraud scores: [0.16117126 0.594132   0.2823311  0.81131816 0.16999882 0.54921687
 0.24787249 0.84144586 0.24423723 0.22457643]


In [38]:
!pip install torch

Defaulting to user installation because normal site-packages is not writeable
